In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, nltk, os, glob
from utils import plot_style, norm01; plot_style();
import torch

### General Notes for markdown intepretation
* each bullet = separate code block
* please appropriately comment
* you'll notice there are 'sections' to this notebook. Try to make each section standalone, i.e., not too dependent on code above, except for the loading of data/model.
* In new markdowns, I may use a different name for the same variable. When possible, update all previous instances to the new name because it is likely better thought out.
* Markdowns will have questions. These are for you to help guide on. After implementing, update markdown with the choice of action.

#### Just build a simple RNN


In [2]:
import torch
import torch.nn as nn

class SimpleRNN(nn.Module):
    def __init__(self, input_size=2, hidden_size=32, output_size=2, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, h = self.rnn(x)       # out: (batch, seq_len, hidden_size)
        out = self.fc(out)         # out: (batch, seq_len, output_size)
        return out, h

In [3]:
model = SimpleRNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Example: sequences of length 10
for epoch in range(500):
    seq = torch.rand(32, 10, 2)        # (batch=32, seq_len=10, features=2)
    pred, _ = model(seq)
    loss = loss_fn(pred, seq)          # target = input (copy task)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

/home/nuttidalab/miniconda3/envs/narrative_map/lib/python3.11/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 0, Loss: 0.6225
Epoch 100, Loss: 0.0661
Epoch 200, Loss: 0.0267
Epoch 300, Loss: 0.0061
Epoch 400, Loss: 0.0010


In [4]:
model.eval()
with torch.no_grad():
    test_seq = torch.rand(2, 10, 2)
    recalled, _ = model(test_seq)
    print("Input:   ", test_seq.squeeze().numpy().round(3))
    print("Recalled:", recalled.squeeze().numpy().round(3))

Input:    [[[0.179 0.413]
  [0.616 0.817]
  [0.288 0.879]
  [0.021 0.307]
  [0.552 0.156]
  [0.358 0.042]
  [0.964 0.63 ]
  [0.    0.438]
  [0.401 0.283]
  [0.668 0.574]]

 [[0.401 0.007]
  [0.138 0.167]
  [0.157 0.109]
  [0.49  0.454]
  [0.798 0.69 ]
  [0.654 0.834]
  [0.102 0.794]
  [0.859 0.167]
  [0.274 0.47 ]
  [0.384 0.462]]]
Recalled: [[[0.215 0.456]
  [0.602 0.803]
  [0.286 0.869]
  [0.031 0.317]
  [0.556 0.165]
  [0.367 0.056]
  [0.945 0.602]
  [0.016 0.453]
  [0.411 0.288]
  [0.663 0.57 ]]

 [[0.437 0.051]
  [0.145 0.179]
  [0.164 0.121]
  [0.493 0.446]
  [0.783 0.671]
  [0.642 0.816]
  [0.105 0.797]
  [0.852 0.172]
  [0.279 0.482]
  [0.389 0.464]]]
